In [ ]:
import requests

cid_link = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsubstructure/smiles/%5BSH%5D/cids/JSON'
response = requests.get(cid_link)
if response.status_code == 200:
    data = response.json()
    cids = data.get('IdentifierList', {}).get('CID', [])
    print(f"Found {len(cids)} CIDs")

In [ ]:
import time

import pandas as pd
import requests
from rdkit import Chem

thiol_pattern = Chem.MolFromSmarts("[C][S;X2;H1]")

# Atomic numbers permitted in CHNOS-only molecules:
# C=6, N=7, O=8, S=16
allowed_atomic_numbers = {6, 7, 8, 16}


def is_chNOS_thiol_under_limit(smiles, max_heavy_atoms=10):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    if len(Chem.GetMolFrags(mol)) != 1:
        return False
    if not mol.HasSubstructMatch(thiol_pattern):
        return False
    if any(atom.GetAtomicNum() not in allowed_atomic_numbers
           for atom in mol.GetAtoms()):
        return False
    heavy_atom_count = mol.GetNumHeavyAtoms()
    return heavy_atom_count <= max_heavy_atoms


# All 808k CIDs in one GET URL is ~7 MB of URL; PubChem drops the connection (the SSLEOFError).
# POST the CIDs in the request body instead, in batches (10k per request takes ~1-2 s).
PROPERTY_URL = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/property/"
                "SMILES,ConnectivitySMILES,IUPACName,MolecularFormula,MolecularWeight,HeavyAtomCount/JSON")
BATCH = 10_000


def fetch_properties(cids, batch=BATCH, retries=3):
    rows = []
    for start in range(0, len(cids), batch):
        chunk = ",".join(map(str, cids[start:start + batch]))
        for attempt in range(retries):
            r = requests.post(PROPERTY_URL, data={"cid": chunk}, timeout=120)
            if r.status_code == 200:
                rows += r.json()["PropertyTable"]["Properties"]
                break
            time.sleep(2 ** attempt)  # 503 = PubChem busy / throttled
        else:
            r.raise_for_status()
        time.sleep(0.25)  # PubChem asks for <= 5 requests/s
        print(f"{min(start + batch, len(cids))}/{len(cids)}", end="\r")
    return pd.DataFrame(rows)


props = fetch_properties(cids)
# cheap prefilter on PubChem's own count, then the RDKit checks
small = props[props["HeavyAtomCount"] <= 10]
thiols = small[small["SMILES"].map(is_chNOS_thiol_under_limit)].reset_index(drop=True)
print(f"\n{len(props)} compounds, {len(small)} with <= 10 heavy atoms, {len(thiols)} CHNOS thiols")
thiols.head()


In [ ]:
# Zwitterion filter: one thiol(ate) for the X-site end (S-) and a basic or
# cationic nitrogen for the A-site end (N+), with no other acidic group, so the
# zwitterion carries one + and one - and is neutral overall. Takes molecules
# drawn neutral (HS...N, from the [SH] search above) and drawn as zwitterions
# ([S-]...[N+], fetched below).
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

if "thiols" not in globals():
    thiols = pd.read_parquet("pubchem_thiols_filtered.parquet")
RDLogger.DisableLog("rdApp.*")  # a few PubChem SMILES fail to parse; they are just skipped

MIN_BONDS_N_S = 3  # 2 = N-C-S hemithioaminals, which fall apart to imine + H2S
MAX_BONDS_N_S = 4  # longer chains can't span A site -> X site within one cage
WEAK_BASES = ["aromatic_N"]  # pyridine / imidazole N (pKa ~5-7): dropped

# --- drawn zwitterions: [S-] on sp3 C joined to an N+ through k atoms -----------
# fastsubstructure ignores formal charges unless MatchCharges=true; without it
# "[S-]" hits PubChem's 1M-record cap. One query per path length (2-9 bonds).
SMARTS_URL = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsubstructure/smarts/"
              "cids/JSON?MatchCharges=true")
zw_cids = set()
for k in range(8):
    r = requests.post(SMARTS_URL, data={"smarts": "[S-][CX4]" + "~*" * k + "~[#7+]"}, timeout=300)
    r.raise_for_status()
    zw_cids |= set(r.json()["IdentifierList"]["CID"])
    time.sleep(0.25)
drawn = fetch_properties(sorted(zw_cids))


def chnos_single_small(smiles, max_heavy_atoms=10):
    mol = Chem.MolFromSmiles(smiles)
    return (mol is not None and len(Chem.GetMolFrags(mol)) == 1
            and mol.GetNumHeavyAtoms() <= max_heavy_atoms
            and all(a.GetAtomicNum() in allowed_atomic_numbers for a in mol.GetAtoms()))


drawn = drawn[drawn["SMILES"].map(chnos_single_small)]
print(f"\n{len(zw_cids)} drawn [S-]...[N+] compounds, {len(drawn)} CHNOS, single fragment, <= 10 heavy atoms")

# --- classify -------------------------------------------------------------------
S = Chem.MolFromSmarts
THIOL_SITE = S("[CX4][$([SX2H1]),$([SX1-])]")  # sp3 C-SH or C-S-; not thioacids or enethiols
CATION_SMARTS = {
    # drawn as a cation: ammonium (protonated or quaternary, e.g. thiocholine), iminium /
    # amidinium, azolium / pyridinium; not nitro, N-oxide, azide/diazo or a protonated amide
    "N+": S("[#7+;!$([#7+]~[#8]);!$([#7+]=,#[#7]);!$([#7+]~[#7]~[#7;X1,X2;!a]);!$([#7+][#6]=[#8])]"),
    # sp3 amine N, not amide/thioamide/sulfonamide/aniline/enamine/hydrazine/hydroxylamine
    "amine": S("[NX3;H2,H1,H0;+0;!$(N[a]);!$(N[#6,#16]=[#7,#8,#16]);!$(N[#6]=[#6]);"
               "!$(N[#7,#8,#16]);!$(NC#N)]"),
    # amidine / guanidine / isothiourea: N-C=N, protonates to an amidinium (like FA+)
    "amidine": S("[NX3;+0;!$(N[#7,#8])][CX3;!a]=[NX2;+0;!$(N[#7,#8])]"),
    # weaker ring bases: imidazole (pKa ~7) and pyridine (~5); not azoles with O/S or >2 N
    "aromatic_N": S("[$([nX2;+0;r5]1:c:[nX3]:c:c:1),$([nX2;+0;r6;!$(n:c:n);!$(n:c:c:n);!$(n:n)])]"),
}
ACID_SMARTS = [S(s) for s in (
    "[CX3](=O)[OX2H1,OX1-]",         # carboxylic acid
    "[CX3](=[O,S])[SX2H1,SX1-]",     # thio-/dithio-acid
    "[CX3](=S)[OX2H1]",
    "[SX4](=O)(=O)[OX2H1,OX1-]",     # sulfonic
    "[SX3](=O)[OX2H1,OX1-]",         # sulfinic
    "[#6]1:[#7]:[#7]:[#7]:[#7]:1",   # tetrazole (acidic NH), any tautomer
)]


def zwitterion_info(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    sh = [m[1] for m in mol.GetSubstructMatches(THIOL_SITE)]
    if len(sh) != 1:                               # exactly one negative site
        return None
    if any(mol.HasSubstructMatch(p) for p in ACID_SMARTS):
        return None
    sites = {k: [m[0] for m in mol.GetSubstructMatches(p)] for k, p in CATION_SMARTS.items()}
    charged = [a.GetIdx() for a in mol.GetAtoms() if a.GetFormalCharge()]
    s_minus = mol.GetAtomWithIdx(sh[0]).GetFormalCharge() == -1
    # allowed: neutral, a lone N+ (S- implied), or the drawn pair N+ / S-
    if len(sites["N+"]) > 1 or set(charged) - set(sites["N+"]) - {sh[0]}:  # e.g. nitro
        return None
    if s_minus and not sites["N+"]:
        return None
    for kind in ("N+", "amidine", "amine", "aromatic_N"):   # strongest base first
        if sites[kind]:
            n = sites[kind]
            break
    else:
        return None
    dm = Chem.GetDistanceMatrix(mol)
    return {"cation": kind,              # strongest basic site found
            "drawn": "zwitterion" if s_minus else ("cation" if sites["N+"] else "neutral"),
            "n_basic_N": len(set(sites["amine"] + sites["amidine"] + sites["aromatic_N"])),
            "bonds_N_S": int(min(dm[i, sh[0]] for i in n))}   # bonds between that N and S


uncharger = rdMolStandardize.Uncharger()


def parent_key(smiles):
    """Neutral form without stereo or isotopes: one key per molecule, however drawn."""
    mol = Chem.MolFromSmiles(smiles)
    for a in mol.GetAtoms():
        a.SetIsotope(0)
        if a.GetSymbol() == "S" and a.GetFormalCharge() == -1:  # S- -> SH, so a quaternary's
            a.SetFormalCharge(0)                                  # thiolate and thiol match
            a.SetNumExplicitHs(a.GetTotalNumHs() + 1)
    mol.UpdatePropertyCache()
    mol = uncharger.uncharge(mol)
    Chem.RemoveStereochemistry(mol)
    return Chem.MolToSmiles(mol)


cands = pd.concat([thiols, drawn], ignore_index=True).drop_duplicates("CID")
info = cands["SMILES"].map(zwitterion_info)
zw = pd.concat([cands[info.notna()].reset_index(drop=True), pd.DataFrame(info.dropna().tolist())], axis=1)
print(f"{len(zw)} of {len(cands)} candidates can form a +1/-1 zwitterion")
zw = zw[zw["bonds_N_S"].between(MIN_BONDS_N_S, MAX_BONDS_N_S) & ~zw["cation"].isin(WEAK_BASES)]
print(f"{len(zw)} with {MIN_BONDS_N_S}-{MAX_BONDS_N_S} bonds N...S and a strong base")
# one row per molecule: drop enantiomers, isotopologues and the same molecule drawn
# neutral and as a zwitterion (keep the lowest CID)
zw["parent"] = zw["SMILES"].map(parent_key)
zwitterions = zw.sort_values("CID").drop_duplicates("parent").reset_index(drop=True)
print(f"{len(zwitterions)} unique molecules")
print(pd.crosstab(zwitterions["cation"], zwitterions["bonds_N_S"], margins=True).to_string())
zwitterions.sort_values(["HeavyAtomCount", "bonds_N_S"]).head(20)

In [ ]:
# A small, varied aliphatic subset, so the organic zwitterions (each run once per
# halide: Cl, Br, I) stay comparable in size to the inorganic halide families
# rather than swamping them. Current data/: MP stable polymorphs ~330 Cl, 178 Br,
# 102 I, 409 F compositions; 1,160 mixed-halide compositions.
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker

N_PICK = 40          # molecules; x 3 halides = 120 samples
N_WELL_KNOWN = 25    # per stratum, pick among the lowest CIDs (older, better-known compounds)
SIMPLE_ONLY = True   # only the cation + thiol (plus OH / ether): a clean amino-alkanethiol class
HALIDES = ["Cl", "Br", "I"]
ALWAYS = {6058: "cysteamine"}  # CIDs always picked; they count toward their stratum's share

UNSTABLE = [S(s) for s in (
    "[CX4]([#7])[OX2H1]",              # hemiaminal
    "[CX4]([OX2H1])[OX2H1]",           # gem-diol
    "[CX4;!R]([#7;!R])[#7;!R]",        # acyclic aminal
    "[CX4]([OX2H1,OX2])[SX2H1,SX1-]",  # hemithioacetal
    "[CX4]([#7])[SX2H1,SX1-]",         # N-C-SH, even when that N is not the cation site
)]
EXTRA_GROUPS = [S(s) for s in (
    "[#6]=[#8]", "[#6]#[#7]", "[#16]=[#8]", "[#6]=[#6]", "[#6]#[#6]", "[#7]~[#7]", "[#7]~[#8]",
)]
CN_DOUBLE = S("[#6]=[#7]")
AMIDINE_C = S("[CX3](~[#7])=[#7]")


def is_simple(mol, cation_class):
    """No functional group besides the cation, the one thiol, OH and ether O, and a
    single basic site (a diamine could take a second proton: net +1, not neutral)."""
    if sum(a.GetAtomicNum() == 16 for a in mol.GetAtoms()) != 1:
        return False
    n_N = sum(a.GetAtomicNum() == 7 for a in mol.GetAtoms())
    if cation_class == "amidine":
        cores = mol.GetSubstructMatches(AMIDINE_C)
        if len(cores) != 1 or n_N != sum(nb.GetAtomicNum() == 7
                                         for nb in mol.GetAtomWithIdx(cores[0][0]).GetNeighbors()):
            return False
    elif n_N != 1:
        return False
    if any(mol.HasSubstructMatch(p) for p in EXTRA_GROUPS):
        return False
    return len(mol.GetSubstructMatches(CN_DOUBLE)) <= (cation_class == "amidine")


def cation_class(smiles, cation):
    """ammonium degree of the N nearest the S (primary ... quaternary) or amidine."""
    mol = Chem.MolFromSmiles(smiles)
    if cation == "amidine":
        return "amidine"
    s_idx = mol.GetSubstructMatch(THIOL_SITE)[1]
    dm = Chem.GetDistanceMatrix(mol)
    n = min((m[0] for m in mol.GetSubstructMatches(CATION_SMARTS[cation])), key=lambda i: dm[i, s_idx])
    atom = mol.GetAtomWithIdx(n)
    if atom.GetHybridization() != Chem.HybridizationType.SP3:
        return "amidine"                                  # amidinium / iminium drawn as N+
    h = atom.GetTotalNumHs() - atom.GetFormalCharge()     # H count of the neutral amine
    return {2: "primary", 1: "secondary", 0: "tertiary"}.get(h, "quaternary") if h >= 0 else "quaternary"


mols = zwitterions["SMILES"].map(Chem.MolFromSmiles)
aliphatic = zwitterions[mols.map(lambda m: not any(a.GetIsAromatic() or a.GetNumRadicalElectrons()
                                                      for a in m.GetAtoms())
                                  and not any(m.HasSubstructMatch(p) for p in UNSTABLE))].copy()
aliphatic["cation_class"] = [cation_class(s, c) for s, c in zip(aliphatic["SMILES"], aliphatic["cation"])]
print(f"{len(aliphatic)} aliphatic, no radicals / hemiaminals / aminals / hemithioacetals")
if SIMPLE_ONLY:
    aliphatic = aliphatic[[is_simple(Chem.MolFromSmiles(s), c)
                           for s, c in zip(aliphatic["SMILES"], aliphatic["cation_class"])]]
    print(f"{len(aliphatic)} with no other functional groups")
print(pd.crosstab(aliphatic["cation_class"], aliphatic["bonds_N_S"], margins=True).to_string())

# Equal share per (cation class, N...S bonds) stratum, the remainder going to the
# strata that have molecules left; inside a stratum, MaxMin on Morgan fingerprints
# among its N_WELL_KNOWN lowest CIDs.
missing = set(ALWAYS) - set(aliphatic["CID"])
assert not missing, f"ALWAYS CIDs not in the aliphatic pool: {missing}"
strata = {k: pd.concat([g[g["CID"].isin(ALWAYS)], g[~g["CID"].isin(ALWAYS)].sort_values("CID").head(N_WELL_KNOWN)])
          for k, g in aliphatic.groupby(["cation_class", "bonds_N_S"])}
quota = {k: int(g["CID"].isin(ALWAYS).sum()) for k, g in strata.items()}
while sum(quota.values()) < N_PICK and any(quota[k] < len(g) for k, g in strata.items()):
    for k in sorted(strata, key=lambda k: len(strata[k])):
        if quota[k] < len(strata[k]) and sum(quota.values()) < N_PICK:
            quota[k] += 1
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
picks = []
for k, g in strata.items():
    if not quota[k]:
        continue
    fps = [fpgen.GetFingerprint(Chem.MolFromSmiles(s)) for s in g["SMILES"]]
    first = [i for i, c in enumerate(g["CID"]) if c in ALWAYS]  # MaxMin starts from these
    idx = (MaxMinPicker().LazyBitVectorPick(fps, len(fps), quota[k], firstPicks=first, seed=0)
           if quota[k] < len(g) else range(len(g)))
    picks.append(g.iloc[list(idx)])
picked = pd.concat(picks).sort_values(["cation_class", "bonds_N_S", "HeavyAtomCount"]).reset_index(drop=True)

aliphatic.to_parquet("pubchem_zwitterions_aliphatic.parquet", index=False)
picked.to_csv("pubchem_zwitterion_picks.csv", index=False)
print(f"\n{len(picked)} picked x {len(HALIDES)} halides = {len(picked) * len(HALIDES)} samples "
      "-> pubchem_zwitterion_picks.csv")
print(pd.crosstab(picked["cation_class"], picked["bonds_N_S"], margins=True).to_string())
picked[["CID", "SMILES", "IUPACName", "HeavyAtomCount", "cation_class", "bonds_N_S"]]

In [ ]:
# Steric fit: most of the aliphatic pool does not fit the 3D cage of the cysteamine
# prototype (the Goldschmidt limit on the A-site cation). The fit screen
#   uv run python3 scripts/dataset/build_zwitterion_perovskites.py --picks pubchem_zwitterions_aliphatic.parquet \
#       --halides Cl --max-expansion 1.10 --screen --jobs 10 --out data/organic_halide_replacement_structures/fit_screen
# builds each molecule into the PbCl2 framework (Cl: the tightest cage) and records the
# isotropic cell expansion needed to clear all steric clashes (2026-11-16: 34 of 1,109
# fit within 1.10, none with more than 7 heavy atoms). Every molecule that fits is
# kept; the Br and I cages are 5% and 12% larger.
from rdkit.Chem import rdMolDescriptors

MAX_EXPANSION = 1.10
LABILE = [S(s) for s in ("[CX4]([#7])[OX2]",)]  # N,O-acetal ethers hydrolyse (cf. hemiaminals)

screen = pd.read_csv("data/organic_halide_replacement_structures/fit_screen/screen.csv")
fit_ok = screen[screen["clash_free"] & (screen["expansion"] <= MAX_EXPANSION + 1e-9)]
fits = aliphatic[aliphatic["CID"].isin(fit_ok["cid"])].merge(
    fit_ok[["cid", "expansion"]].rename(columns={"cid": "CID", "expansion": "expansion_Cl"}), on="CID")
fmols = fits["SMILES"].map(Chem.MolFromSmiles)
keep = ~fmols.map(lambda m: any(m.HasSubstructMatch(p) for p in LABILE)
                  or rdMolDescriptors.CalcNumRings(m) > 1)  # fused bicyclo[1.1.0]/[2.1.0]: too strained
print(f"{len(fits)} of {len(screen)} fit with expansion <= {MAX_EXPANSION} (Cl); "
      f"dropped {(~keep).sum()} labile / strained:", ", ".join(fits.loc[~keep, "SMILES"]))
fit_picks = fits[keep].sort_values(["expansion_Cl", "HeavyAtomCount", "CID"]).reset_index(drop=True)
assert set(ALWAYS) <= set(fit_picks["CID"])
fit_picks.to_csv("pubchem_zwitterion_fit_picks.csv", index=False)
print(f"{len(fit_picks)} molecules x {len(HALIDES)} halides = {len(fit_picks) * len(HALIDES)} samples "
      "-> pubchem_zwitterion_fit_picks.csv")
print(pd.crosstab(fit_picks["cation_class"], fit_picks["bonds_N_S"], margins=True).to_string())
fit_picks[["CID", "SMILES", "IUPACName", "HeavyAtomCount", "cation_class", "bonds_N_S", "expansion_Cl"]]